# Buy Low, Sell High (It's Harder Than It Sounds)
### A hands-on introduction to ML for trading — GameStop vs. the S&P 500 vs. Bitcoin

This notebook accompanies the lecture. You will:

1. **Explore the data** and discover why the *way* you measure returns changes the story completely
2. **Build online-learning strategies** (Follow-The-Leader and friends) on the full S&P 500
3. **Fit classical forecasting models** (AR, MA, ARIMA, seasonal ARIMA) and watch them diverge from reality
4. **Train RNNs and a GRU** — and compare them against an "embarrassingly simple" linear model (Zeng et al., 2022)

Along the way we keep score the same way for every strategy: **invest \$1 every day, sell at the end of the day**, and report total profit and the **Sharpe ratio**.

> ⚖️ **Legal disclaimer:** this notebook is for education only. Nothing here is financial advice.
> Remember the lecture: when you trade, you are betting that *you* are right and the *crowd* is wrong.
> The crowd guessed the weight of Galton's ox almost perfectly. Plan accordingly.

> 🖥️ **Runtime:** works on any runtime; a free GPU (`Runtime → Change runtime type → T4 GPU`) makes Part 4 a bit faster. Total runtime is only a few minutes either way.


## Setup

One small install, shared imports, and two helpers we will use all notebook long:

- `sharpe(daily_pnl)` — the annualized **Sharpe ratio**: average daily profit divided by its standard deviation, scaled to a year ($\times \sqrt{252}$). It asks: *how much return am I getting per unit of risk?* A Sharpe of 1 is good, 2 is great, and if you see 5+ you should suspect a bug (or an oracle).
- `pct(x)` — pretty-print a return as a percentage.


In [ ]:
%pip install -q --upgrade yfinance

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams.update({
    "figure.figsize": (12, 5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

RNG = np.random.default_rng(0)
TRADING_DAYS = 252


def sharpe(daily_pnl):
    """Annualized Sharpe ratio of a series of daily profits per $1 invested."""
    daily_pnl = pd.Series(daily_pnl).dropna()
    if len(daily_pnl) < 2 or daily_pnl.std() == 0:
        return float("nan")
    return float(daily_pnl.mean() / daily_pnl.std() * np.sqrt(TRADING_DAYS))


def pct(x):
    return f"{100 * x:+,.1f}%"


---
# Part 1 — Data exploration

We compare three very different assets:

| Ticker | Asset | Personality |
|---|---|---|
| `GME` | GameStop | meme-stock chaos (regime changes!) |
| `^GSPC` | S&P 500 index | "the market" |
| `BTC-USD` | Bitcoin | high-variance, trades 24/7 |

**👉 Add any ticker you like to the list below** (e.g. `"TSLA"`, `"NVDA"`, `"DOGE-USD"`) and re-run the notebook — everything downstream adapts automatically.

*(One caveat: we keep only days where **all** assets traded, so crypto weekends are dropped, and a ticker that IPO'd recently will shorten the whole sample.)*


In [ ]:
# A — download daily open/close prices with yfinance
TICKERS = ["GME", "^GSPC", "BTC-USD"]   # <-- add your own tickers here!
START = "2019-01-01"

raw = yf.download(TICKERS, start=START, auto_adjust=True, progress=False)

# keep only days where every asset traded (crypto trades on weekends, stocks don't)
opens = raw["Open"].dropna()
closes = raw["Close"].dropna()
opens, closes = opens.align(closes, join="inner", axis=0)

print(f"{len(closes)} shared trading days, {closes.index[0].date()} -> {closes.index[-1].date()}")
closes.tail()


## B — Strategy 1: buy \$1 on day one and hold

Normalize every asset to "what is my \$1 from day one worth today?" so they share one axis.


In [ ]:
growth = closes / closes.iloc[0]   # value today of $1 invested on day one

ax = growth.plot(logy=True, title="Growth of $1 invested on day one (buy & hold)")
ax.set_ylabel("portfolio value ($, log scale)")
plt.show()

print(f"Total buy-&-hold return since {growth.index[0].date()}:")
for t in growth.columns:
    print(f"  {t:>8}: {pct(growth[t].iloc[-1] - 1)}")


## C — Strategy 2: invest \$1 *every day*, sell at the close

Every day we put exactly \$1 in at the open and cash out at the close. Each day is an
independent little bet, so the cumulative profit is just the **sum** of daily open→close
returns — no compounding. This is the scoring rule we will use for *every* strategy in
this notebook, because it makes wildly different strategies directly comparable.


In [ ]:
day_ret = closes / opens - 1   # profit per $1: buy the open, sell the close
cum_profit = day_ret.cumsum()

ax = cum_profit.plot(title="Cumulative profit: $1 in at the open, out at the close, every day")
ax.set_ylabel("total profit ($)")
plt.show()

print(f"{'asset':>8} | {'total profit':>12} | {'avg daily return':>16} | Sharpe")
print("-" * 55)
for t in day_ret.columns:
    print(f"{t:>8} | {'$' + format(cum_profit[t].iloc[-1], ',.2f'):>12} | "
          f"{pct(day_ret[t].mean()):>16} | {sharpe(day_ret[t]):>6.2f}")


## D — Which measurement can you trust?

Here is the trap: **buy & hold returns depend enormously on the day you start.**
Buy GME in 2019 and you are a genius; buy it in February 2021 and you are not.

Below we slide the start date of each strategy forward one quarter at a time and
re-measure its annualized return. Watch which line jumps around and which stays flat.


In [ ]:
# try every start date (quarterly steps over the first 80% of the sample)
start_dates = closes.index[: int(len(closes) * 0.8) : 63]

bh = pd.DataFrame({s: (closes.iloc[-1] / closes.loc[s:].iloc[0]) **
                      (TRADING_DAYS / len(closes.loc[s:])) - 1 for s in start_dates}).T
dd = pd.DataFrame({s: day_ret.loc[s:].mean() * TRADING_DAYS for s in start_dates}).T

fig, axes = plt.subplots(1, len(closes.columns), figsize=(15, 4), sharex=True)
for ax, t in zip(np.atleast_1d(axes), closes.columns):
    ax.plot(bh.index, 100 * bh[t], lw=2, label="buy & hold (annualized)")
    ax.plot(dd.index, 100 * dd[t], lw=2, label="$1/day (annualized)")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(t)
    ax.set_ylabel("annualized return (%)")
    ax.tick_params(axis="x", rotation=45)
np.atleast_1d(axes)[0].legend()
fig.suptitle("Same strategy, different start date — which measurement is robust?", y=1.04)
plt.tight_layout()
plt.show()


**Takeaway.** Buy & hold returns are dominated by a single number — the entry price —
so moving the start date changes the story completely. The \$1/day measurement averages
over thousands of independent entries, so it barely moves.

This is the lecture's **implicit p-hacking** point: if a backtest lets you pick the start
date, you can make almost any strategy look brilliant. Whenever someone shows you a
backtest, ask: *what happens if I shift the start date by six months?*


---
# Part 2 — Online learning: Follow-The-Leader on the full S&P 500

Now we widen the playing field from 3 assets to **all ~500 stocks in the S&P 500**, and
play a game from online learning theory. Every morning you must put your \$1 on exactly
one stock (or spread it across several). At the close you cash out and the day's
*leader board* is revealed.

We will compare:

- **D. Random** — pick a stock uniformly at random each day
- **E. Oracle** — cheat: pick the stock that will do best *today* (needs a time machine)
- **F. FTL** — *Follow-The-Leader*: pick the stock that did best **yesterday**
- **F'. Hedge** — *regularized* FTL: spread the \$1 over all stocks, weighting each by
  $e^{\eta \cdot (\text{its cumulative return so far})}$ (multiplicative weights)

⚠️ **Survivorship bias alert** (lecture section 4!): we use *today's* index members, so
companies that crashed out of the index never appear in our data. Our backtest is
slightly rosier than reality before we even start.


In [ ]:
# get the current S&P 500 member list from Wikipedia, then bulk-download 2 years of prices
import io
import requests

WIKI = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = requests.get(WIKI, headers={"User-Agent": "Mozilla/5.0 (educational notebook)"}, timeout=30).text
table = pd.read_html(io.StringIO(html))[0]
snp_tickers = sorted(table["Symbol"].str.replace(".", "-", regex=False))
print(f"{len(snp_tickers)} tickers: {snp_tickers[:8]} ...")

snp_raw = yf.download(snp_tickers, period="2y", auto_adjust=True, progress=False)

# open->close return for every stock, every day; drop stocks with patchy data
snp_ret = (snp_raw["Close"] / snp_raw["Open"] - 1)
snp_ret = snp_ret.dropna(axis=1, thresh=int(0.95 * len(snp_ret))).fillna(0.0)
R = snp_ret.to_numpy()
n_days, n_stocks = R.shape
print(f"return matrix: {n_days} days x {n_stocks} stocks")


### D — The dart-throwing monkey

Pick one stock uniformly at random every day. We run 5 monkeys to see how much luck matters.


In [ ]:
fig, ax = plt.subplots()
paths = []
for monkey in range(5):
    picks = RNG.integers(0, n_stocks, size=n_days)
    pnl = R[np.arange(n_days), picks]
    paths.append(pnl)
    ax.plot(snp_ret.index, pnl.cumsum(), color="gray", alpha=0.5, lw=1)

random_pnl = pd.Series(np.mean(paths, axis=0), index=snp_ret.index, name="random monkey")
ax.plot(snp_ret.index, random_pnl.cumsum(), lw=2.5, label="average of 5 monkeys")
ax.set_title("$1/day on one uniformly random S&P 500 stock")
ax.set_ylabel("cumulative profit ($)")
ax.legend()
plt.show()
print(f"average monkey: total {'$' + format(random_pnl.sum(), ',.2f')}, Sharpe {sharpe(random_pnl):.2f}")


### E — The oracle

Suppose you could see **today's** returns before the market opens and always pick the
day's single best stock. This is the *comparator* that online learning measures regret
against — and it shows how much money is theoretically on the table every single day.


In [ ]:
oracle_pnl = pd.Series(R.max(axis=1), index=snp_ret.index, name="oracle")

ax = oracle_pnl.cumsum().plot(title="The forward-looking oracle: always holds today's best stock")
ax.set_ylabel("cumulative profit ($)")
plt.show()

best = snp_ret.columns[R.argmax(axis=1)]
print(f"oracle: avg daily return {pct(oracle_pnl.mean())}, Sharpe {sharpe(oracle_pnl):.2f}")
print(f"its 5 favorite stocks: {dict(pd.Series(best).value_counts().head(5))}")


### F — Follow-The-Leader (and its regularized cousin)

No time machine now. **FTL** bets everything on *yesterday's* best stock — the most
literal version of "momentum". **Hedge** (multiplicative weights) is the regularized
version from lecture: it never goes all-in, it just tilts its \$1 toward stocks that have
been doing well cumulatively. Try a few values of `eta`: $\eta = 0$ is the uniform
portfolio, $\eta \to \infty$ approaches all-in FTL on the cumulative leader.


In [ ]:
# F1 — FTL: all-in on whichever stock did best yesterday
ydy_best = R[:-1].argmax(axis=1)            # yesterday's winner...
ftl_pnl = pd.Series(R[1:][np.arange(n_days - 1), ydy_best],
                    index=snp_ret.index[1:], name="FTL (yesterday's best)")

# F2 — Hedge: weight each stock by exp(eta * cumulative return so far)
eta = 5.0
cum = np.vstack([np.zeros(n_stocks), R.cumsum(axis=0)[:-1]])   # returns known *before* each day
W = np.exp(eta * (cum - cum.max(axis=1, keepdims=True)))
W /= W.sum(axis=1, keepdims=True)
hedge_pnl = pd.Series((W * R).sum(axis=1), index=snp_ret.index, name=f"Hedge (eta={eta:g})")

# baseline: spread $1 evenly over every stock, every day
uniform_pnl = pd.Series(R.mean(axis=1), index=snp_ret.index, name="1/N uniform")

strategies = pd.concat([random_pnl, uniform_pnl, ftl_pnl, hedge_pnl], axis=1)

ax = strategies.cumsum().plot(title="$1/day strategies on the S&P 500 field")
ax.set_ylabel("cumulative profit ($)")
plt.show()

summary = pd.DataFrame({
    "total profit ($)": strategies.sum().round(2),
    "avg daily return (%)": (100 * strategies.mean()).round(3),
    "Sharpe": strategies.apply(sharpe).round(2),
})
summary.loc["oracle (cheating)"] = [oracle_pnl.sum().round(2),
                                    (100 * oracle_pnl.mean()).round(3),
                                    round(sharpe(oracle_pnl), 2)]
summary


**Takeaways.**

- The oracle is absurdly far above everything else: *information about the future is the
  whole game*. Online learning's "regret" measures how far behind it you are.
- All-in FTL is jumpy: yesterday's single best stock is usually a one-day fluke that
  **mean-reverts**, so chasing it buys volatility, not signal.
- Hedge is smoother — regularization stops one lucky stock from grabbing the whole
  portfolio — and lands near the uniform portfolio. On a roughly efficient market,
  beating 1/N is genuinely hard.

### One more thing: fees

Every trade costs something (commission, spread, slippage). Our strategies trade **every
single day**, so even a tiny per-trade fee compounds into a big haircut. Watch what a
realistic 5 basis points (0.05%) per day does:


In [ ]:
FEE = 0.0005   # 5 bp per round-trip — optimistic for a retail trader
after_fees = strategies - FEE

fig, ax = plt.subplots()
for col, color in zip(strategies.columns, plt.rcParams["axes.prop_cycle"].by_key()["color"]):
    ax.plot(strategies.index, strategies[col].cumsum(), color=color, lw=2, label=f"{col}")
    ax.plot(after_fees.index, after_fees[col].cumsum(), color=color, lw=1.5, ls="--")
ax.set_title("Solid = before fees, dashed = after a 0.05% daily fee")
ax.set_ylabel("cumulative profit ($)")
ax.legend()
plt.show()

pd.DataFrame({"Sharpe before fees": strategies.apply(sharpe).round(2),
              "Sharpe after fees": after_fees.apply(sharpe).round(2)})


---
# Part 3 — Classical forecasting: AR, MA, ARIMA, S-ARIMA

Different game now: instead of *reacting* to prices, we try to **predict** them.

The cardinal rule of evaluating forecasters: pick a **cutoff date**, train only on data
*before* it, then compare the model's predictions against what *actually happened* after.
The shaded bands are the models' own 95% confidence intervals — watch how honest (or not)
they are about their uncertainty.

We forecast the S&P 500 index level, training on everything before the cutoff (the last
~15% of days is held out as the test set).


In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

price = closes["^GSPC"].rename("S&P 500")
CUTOFF = price.index[int(len(price) * 0.85)]
train, test = price.loc[:CUTOFF], price.loc[CUTOFF:][1:]
print(f"train: {train.index[0].date()} -> {train.index[-1].date()}  ({len(train)} days)")
print(f"test : {test.index[0].date()} -> {test.index[-1].date()}  ({len(test)} days)")


def forecast_plot(models, title, context_days=500):
    """Fit results -> forecast over the test window, plotted against reality."""
    fig, ax = plt.subplots()
    ax.plot(train.index[-context_days:], train.iloc[-context_days:],
            color="black", lw=1, label="train (actual)")
    ax.plot(test.index, test.to_numpy(), color="gray", lw=1.5, label="test (actual)")
    rmses = {}
    for name, res in models.items():
        fc = res.get_forecast(len(test))
        mean = np.asarray(fc.predicted_mean)
        ci = np.asarray(fc.conf_int())
        ax.plot(test.index, mean, lw=2, label=name)
        ax.fill_between(test.index, ci[:, 0], ci[:, 1], alpha=0.12)
        rmses[name] = float(np.sqrt(np.mean((mean - test.to_numpy()) ** 2)))
    rmses["naive (yesterday's price)"] = float(np.sqrt(np.mean((train.iloc[-1] - test.to_numpy()) ** 2)))
    ax.axvline(CUTOFF, color="red", ls="--", lw=1, label="cutoff")
    ax.set_title(title)
    ax.set_ylabel("index level")
    ax.legend(loc="upper left")
    plt.show()
    return pd.Series(rmses, name="test RMSE").round(1).to_frame()


## G — AR: autoregression

$\text{AR}(p)$ predicts tomorrow as a weighted sum of the last $p$ days:
$\;y_{t} = c + \varphi_1 y_{t-1} + \dots + \varphi_p y_{t-p} + \varepsilon_t$.

On stock *prices* the fitted $\varphi_1$ comes out almost exactly 1 — the model
rediscovers that a stock price is (nearly) a **random walk**, and its best forecast is
"about the same as yesterday, forever."


In [ ]:
ar_models = {f"AR({p})": ARIMA(train.to_numpy(), order=(p, 0, 0)).fit() for p in (1, 5)}
forecast_plot(ar_models, "AR models: trained before the cutoff, forecasting after")


## H — MA: moving average

$\text{MA}(q)$ models today as the mean plus the last $q$ random *shocks*:
$\;y_t = \mu + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \dots + \theta_q \varepsilon_{t-q}$.

An MA model has only $q$ days of memory. Once the forecast horizon passes $q$ days, it
has nothing left to say except "the historical average" — watch it snap back to the mean
of the whole training set. (Spectacularly wrong, and the confidence band knows it.)


In [ ]:
ma_models = {f"MA({q})": ARIMA(train.to_numpy(), order=(0, 0, q)).fit() for q in (1, 5)}
forecast_plot(ma_models, "MA models forget everything after q days")


## I — ARIMA: putting it together

$\text{ARIMA}(p, d, q)$ = AR($p$) + MA($q$) fitted on the series **differenced** $d$
times. With $d=1$ we model daily *changes* instead of levels — the right call for a
random-walk-like series. Try a few configurations; feel free to edit the list.

Notice that every configuration collapses to nearly the same forecast: a flat-ish line
continuing from the last training price, with uncertainty growing like $\sqrt{t}$.
That *is* the random walk model. The RMSE table tells the punchline: none of them
meaningfully beat the naive "tomorrow = yesterday" forecast.


In [ ]:
configs = [(1, 1, 1), (2, 1, 2), (5, 1, 0)]
arima_models = {f"ARIMA{c}": ARIMA(train.to_numpy(), order=c).fit() for c in configs}
forecast_plot(arima_models, "ARIMA: every config converges to 'random walk with noise'")


## J — Seasonal ARIMA on quarterly data

Markets have plausible seasonal stories (earnings every quarter, the "January effect",
"sell in May"...). S-ARIMA adds a seasonal AR/MA term at a fixed lag — here we resample
to **quarterly** closes and use a season length of 4 (one year).


In [ ]:
q_price = price.resample("QE").last()
q_train = q_price.loc[:CUTOFF]
q_test = q_price.loc[CUTOFF:][1:]

sarima = SARIMAX(q_train.to_numpy(), order=(1, 1, 1),
                 seasonal_order=(1, 1, 1, 4)).fit(disp=False)
fc = sarima.get_forecast(len(q_test))
mean, ci = np.asarray(fc.predicted_mean), np.asarray(fc.conf_int())

fig, ax = plt.subplots()
ax.plot(q_train.index, q_train.to_numpy(), "o-", color="black", lw=1, label="train (quarterly)")
ax.plot(q_test.index, q_test.to_numpy(), "o-", color="gray", lw=1.5, label="test (actual)")
ax.plot(q_test.index, mean, "o-", lw=2, label="S-ARIMA(1,1,1)x(1,1,1,4)")
ax.fill_between(q_test.index, ci[:, 0], ci[:, 1], alpha=0.12)
ax.axvline(CUTOFF, color="red", ls="--", lw=1, label="cutoff")
ax.set_title("Quarterly S-ARIMA forecast vs. reality")
ax.set_ylabel("index level")
ax.legend(loc="upper left")
plt.show()


**Takeaway.** Classical models are honest about one big truth: at a daily/quarterly
horizon, index prices are very close to a random walk. The models do not fail because
they are dumb — they fail because **the predictable part of the signal is tiny**, and
they correctly tell you so via exploding confidence intervals. Any model (or person)
showing you a confident long-horizon price forecast is hiding the error bars.


---
# Part 4 — Neural networks: RNN, GRU... and an embarrassingly simple linear model

Finally, deep learning. We frame forecasting as supervised learning:

> **Input:** the last 30 days of (standardized) log-returns. **Target:** tomorrow's log-return.

Same discipline as Part 3 — train strictly before the cutoff, evaluate strictly after.
We train three models:

- **J. RNN** — a vanilla recurrent network
- **K. GRU** — a gated RNN (LSTM's leaner sibling)
- **L. Linear** — one `nn.Linear` layer, the "embarrassingly simple" baseline in the
  spirit of Zeng et al. (2022), *"Are Transformers Effective for Time Series Forecasting?"*

Each model is small and trains in seconds (GPU helps, but isn't required).


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"training on: {device}")

WINDOW = 30
log_ret = np.log(price / price.shift(1)).dropna()

X_all = np.lib.stride_tricks.sliding_window_view(log_ret.to_numpy(), WINDOW)[:-1]
y_all = log_ret.to_numpy()[WINDOW:]
dates = log_ret.index[WINDOW:]

split = dates.searchsorted(CUTOFF)
mu, sd = X_all[:split].mean(), X_all[:split].std()    # standardize using TRAIN stats only
Xn, yn = (X_all - mu) / sd, (y_all - mu) / sd

Xtr = torch.tensor(Xn[:split], dtype=torch.float32).unsqueeze(-1).to(device)
ytr = torch.tensor(yn[:split], dtype=torch.float32).to(device)
Xte = torch.tensor(Xn[split:], dtype=torch.float32).unsqueeze(-1).to(device)
print(f"{len(Xtr)} training windows, {len(Xte)} test windows, {WINDOW} days each")


In [ ]:
class RecurrentForecaster(nn.Module):
    def __init__(self, cell=nn.RNN, hidden=32):
        super().__init__()
        self.rnn = cell(input_size=1, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):                      # x: (batch, WINDOW, 1)
        out, _ = self.rnn(x)
        return self.head(out[:, -1]).squeeze(-1)


class JustLinear(nn.Module):
    # Zeng et al. (2022): one linear layer from the raw window to the prediction
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(WINDOW, 1)

    def forward(self, x):
        return self.lin(x.squeeze(-1)).squeeze(-1)


def train_model(model, epochs=20, batch=128, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for _ in range(epochs):
        perm = torch.randperm(len(Xtr), device=device)
        total = 0.0
        for i in range(0, len(Xtr), batch):
            idx = perm[i:i + batch]
            loss = nn.functional.mse_loss(model(Xtr[idx]), ytr[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        losses.append(total / len(Xtr))
    return model, losses


## J & K & L — train all three

(Each takes a few seconds. If this cell takes more than a minute, something is wrong.)


In [ ]:
%%time
models, curves = {}, {}
for name, net in [("RNN", RecurrentForecaster(nn.RNN)),
                  ("GRU", RecurrentForecaster(nn.GRU)),
                  ("Linear", JustLinear())]:
    models[name], curves[name] = train_model(net)
    print(f"{name:>7}: final train MSE {curves[name][-1]:.4f}")

fig, ax = plt.subplots()
for name, c in curves.items():
    ax.plot(range(1, len(c) + 1), c, lw=2, label=name)
ax.set_title("Training loss")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE (standardized returns)")
ax.legend()
plt.show()


## Did we learn anything real?

Two checks, in increasing order of honesty:

1. **Test MSE** vs. the dumbest possible baseline: *always predict tomorrow's return is
   0* (i.e. "prices are a random walk").
2. **A trading backtest**: each test day, go long \$1 if the model predicts an up day,
   short \$1 if down — versus just holding the index long every day.


In [ ]:
test_dates = dates[split:]
actual = y_all[split:]                       # true next-day log-returns (test set)

with torch.no_grad():
    preds = {name: (m(Xte).cpu().numpy() * sd + mu) for name, m in models.items()}

rows = {"always predict 0": {"test MSE (x1e6)": 1e6 * np.mean(actual ** 2),
                             "sign accuracy": np.mean(actual > 0)}}
for name, p in preds.items():
    rows[name] = {"test MSE (x1e6)": 1e6 * np.mean((p - actual) ** 2),
                  "sign accuracy": np.mean(np.sign(p) == np.sign(actual))}
display(pd.DataFrame(rows).T.round(3))

simple_ret = np.exp(actual) - 1              # log-return -> simple return
backtest = pd.DataFrame({"long the index": simple_ret}, index=test_dates)
for name, p in preds.items():
    backtest[name] = np.sign(p) * simple_ret

ax = backtest.cumsum().plot(title="Test-period backtest: $1/day, long or short by predicted sign")
ax.set_ylabel("cumulative profit ($)")
plt.show()

pd.DataFrame({"total profit ($)": backtest.sum().round(3),
              "Sharpe": backtest.apply(sharpe).round(2)})


## Final takeaways

1. **The linear model keeps up with the RNNs.** That is Zeng et al.'s point: on noisy
   financial series, architectural sophistication adds little, because there is hardly
   any signal to be sophisticated *about*. Transformers (and GRUs) are not all you need.
2. **Everyone's predictions hover near zero** — the networks mostly learn "tomorrow's
   return is unpredictable," which is the *correct* lesson, just an unprofitable one.
3. **A backtest Sharpe near (or below) the long-the-index baseline** means your model
   has no real *alpha* — it is repackaging market *beta*. Disaggregate them before
   celebrating (lecture, section 3).
4. And everything from lecture section 4 still applies on top: our backtest has **no
   fees, no slippage, no market impact**, uses survivorship-biased data, one cutoff
   date (try moving it — that is signal dropout / placebo testing!), and the test
   period contains exactly **one** regime. On paper is the easy part.

### Where to go from here

- Add your own tickers in Part 1 and re-run everything.
- In Part 2, tune `eta` — can you beat 1/N? Are you sure it's not noise?
- In Part 3, move `CUTOFF` and watch every conclusion wobble.
- In Part 4, swap `nn.GRU` for `nn.LSTM` (one-word change), add features
  (volume? volatility?), or predict 5 days ahead instead of 1.

> ⚖️ One last time: this is a *classroom*, not a brokerage. The fastest way to use this
> notebook to make money is to not trade with it.
